# Development of an Artificial Neural Network (ANN) for Binary Classification
## Benchmark on Scikit-Learn `make_moons` Dataset

### Objective
Develop, train, evaluate, and visualize a multi-layer **Artificial Neural Network (ANN)** in PyTorch to solve non-linear binary classification on the two-moon dataset (`make_moons`). We also compare its performance against a linear baseline classifier to highlight the role of non-linear activations.

### 1. Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
print('PyTorch Version:', torch.__version__)

### 2. Generate and Explore the `make_moons` Dataset
The dataset consists of two interleaving half-circles with Gaussian noise.

In [ ]:
X, y = make_moons(n_samples=1500, noise=0.20, random_state=42)

plt.figure(figsize=(8, 6), dpi=120)
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='#3b82f6', label='Moon 0', edgecolors='k', s=40, alpha=0.8)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='#f43f5e', label='Moon 1', edgecolors='k', s=40, alpha=0.8)
plt.title('Synthetic make_moons Dataset (N=1500, Noise=0.20)', fontsize=13, fontweight='bold')
plt.xlabel('Feature X1')
plt.ylabel('Feature X2')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 3. Data Splitting & Feature Standardization
We split the data into **70% Training**, **15% Validation**, and **15% Test** sets with stratified sampling. To avoid data leakage, `StandardScaler` is fitted solely on the training partition.

In [ ]:
# Stratified train-val-test split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15/0.85, random_state=42, stratify=y_train_val)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# PyTorch DataLoaders
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)), batch_size=32, shuffle=False)

print(f'Train samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}')

### 4. Baseline Linear Perceptron vs. Deep ANN Architecture

In [ ]:
# Baseline Linear Model (Logistic Regression Perceptron)
class LinearBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, x):
        return self.fc(x)

# Deep Multi-Layer Artificial Neural Network
class DeepMoonANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.1),
            nn.Linear(16, 8),
            nn.LeakyReLU(0.1),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.net(x)

baseline_model = LinearBaseline()
ann_model = DeepMoonANN()
print('Baseline parameters:', sum(p.numel() for p in baseline_model.parameters()))
print('Deep ANN parameters: ', sum(p.numel() for p in ann_model.parameters()))

### 5. Training Loop with Adam, Learning Rate Scheduler, and Early Stopping

In [ ]:
import copy

def train_ann(model, train_loader, val_loader, epochs=120, lr=0.01):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_weights = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    
    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        t_loss, t_corr, total = 0.0, 0, 0
        for bx, by in train_loader:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * bx.size(0)
            t_corr += ((torch.sigmoid(out) >= 0.5) == by).sum().item()
            total += bx.size(0)
            
        # Validation
        model.eval()
        v_loss, v_corr, v_total = 0.0, 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                out = model(bx)
                loss = criterion(out, by)
                v_loss += loss.item() * bx.size(0)
                v_corr += ((torch.sigmoid(out) >= 0.5) == by).sum().item()
                v_total += bx.size(0)
                
        epoch_v_loss = v_loss / v_total
        scheduler.step(epoch_v_loss)
        
        history['train_loss'].append(t_loss / total)
        history['val_loss'].append(epoch_v_loss)
        history['train_acc'].append(t_corr / total)
        history['val_acc'].append(v_corr / v_total)
        
        if epoch_v_loss < best_val_loss:
            best_val_loss = epoch_v_loss
            best_weights = copy.deepcopy(model.state_dict())
            
    model.load_state_dict(best_weights)
    return model, history

print('Training Baseline...')
baseline_model, _ = train_ann(baseline_model, train_loader, val_loader, epochs=60, lr=0.02)
print('Training Deep ANN...')
ann_model, ann_history = train_ann(ann_model, train_loader, val_loader, epochs=120, lr=0.01)

### 6. Quantitative Evaluation on Test Set

In [ ]:
def evaluate(model, X_arr, y_arr):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_arr, dtype=torch.float32))
        probs = torch.sigmoid(logits).numpy().flatten()
    preds = (probs >= 0.5).astype(int)
    return {
        'acc': accuracy_score(y_arr, preds),
        'prec': precision_score(y_arr, preds),
        'rec': recall_score(y_arr, preds),
        'f1': f1_score(y_arr, preds),
        'auc': roc_auc_score(y_arr, probs),
        'probs': probs,
        'preds': preds
    }

base_eval = evaluate(baseline_model, X_test_scaled, y_test)
ann_eval = evaluate(ann_model, X_test_scaled, y_test)

print(f"Baseline Linear Model  -> Accuracy: {base_eval['acc']*100:.2f}%, ROC-AUC: {base_eval['auc']:.4f}")
print(f"Deep Artificial Neural -> Accuracy: {ann_eval['acc']*100:.2f}%, ROC-AUC: {ann_eval['auc']:.4f}")

### 7. Decision Boundary Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
x_min, x_max = X_test_scaled[:, 0].min() - 0.5, X_test_scaled[:, 0].max() + 0.5
y_min, y_max = X_test_scaled[:, 1].min() - 0.5, X_test_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

for model, ax, name, acc in [(baseline_model, axes[0], 'Linear Baseline', base_eval['acc']),
                             (ann_model, axes[1], 'Deep Neural Network', ann_eval['acc'])]:
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).numpy().reshape(xx.shape)
    ax.contourf(xx, yy, probs, levels=25, cmap='RdBu_r', alpha=0.65)
    ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
    ax.scatter(X_test_scaled[y_test == 0, 0], X_test_scaled[y_test == 0, 1], c='#1d4ed8', edgecolors='w', label='Class 0')
    ax.scatter(X_test_scaled[y_test == 1, 0], X_test_scaled[y_test == 1, 1], c='#be123c', edgecolors='w', label='Class 1')
    ax.set_title(f"{name} (Accuracy: {acc*100:.2f}%)", fontweight='bold')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.show()